# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("PRICE_DATA"))


../../05_src/data/prices/


In [8]:
%pip install "dask[dataframe]" pyarrow

     ---------------------------------------- 1.5/1.5 MB 2.0 MB/s eta 0:00:00
     --------------------------------------- 27.8/27.8 MB 16.4 MB/s eta 0:00:00
     ---------------------------------------- 221.7/221.7 kB ? eta 0:00:00
     ---------------------------------------- 60.8/60.8 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [6]:
import os
from glob import glob

# Write your code below.
price_data = os.getenv("PRICE_DATA")
print(price_data)

parquet_files = glob(os.path.join(price_data, "**", "*.parquet"), recursive=True)
print("Number of files:", len(parquet_files))





../../05_src/data/prices/
Number of files: 2045


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [10]:
# Write your code below.
import dask.dataframe as dd

prices = dd.read_parquet(parquet_files).rename(columns={"Adj Close": "Adj_Close"}).set_index("ticker")
def features(x):
    x = x.sort_values("Date") 
    return x.assign(Close_lag_1 = x.Close.shift(1),
Adj_Close_lag_1 = x.Adj_Close.shift(1),
returns = x.Close/x.Close.shift(1) - 1, hi_lo_range = x.High - x.Low)

dd_feat = prices.groupby("ticker", group_keys = False).apply(
    features, meta = features(prices._meta)
)



+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [11]:
# Write your code below.
df = dd_feat.compute().sort_values(["ticker", "Date"])
df["returns_ma_10"] = df.groupby("ticker")["returns"].transform(
    lambda x:
x.rolling(10).mean()
)
df.head(15)

,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
ticker,,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30,NaN
ACN,2001-07-26,14.95,14.99,14.50,14.50,10.900705,6335300.0,ACN.csv,2001,14.95,11.238999,-0.030100,0.49,NaN
ACN,2001-07-27,14.51,14.59,14.50,14.51,10.908223,3524000.0,ACN.csv,2001,14.50,10.900705,0.000690,0.09,NaN
ACN,2001-07-30,14.50,14.78,14.50,14.70,11.051059,3654300.0,ACN.csv,2001,14.51,10.908223,0.013094,0.28,NaN
ACN,2001-07-31,14.71,15.01,14.60,14.96,11.246520,1429000.0,ACN.csv,2001,14.70,11.051059,0.017687,0.41,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, Dask can also calculate moving averages. It is better for large datasets, while pandas is simpler for data that fits in memory.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.